#   Centroidal Floating-Base Dynamics

## Learning objectives

By the end of the lab you should be able to:

- compute the robot center of mass from link inertial data;
- construct a centroidal change of generalized velocity coordinates;
- verify that the transformed mass matrix becomes block diagonal;
- inspect the form of the gravity vector in centroidal coordinates.



## Background

A conventional floating-base frame \(B\) is often attached to the trunk because it preserves useful kinematic sparsity. A different velocity parametrization can instead use the center-of-mass linear velocity and an average angular velocity. With this choice, the floating-base part of the inertia can be decoupled from the actuated joints.

The coordinate transform used in the lab is

$
{}_G T_B =
\begin{bmatrix}
{}_G X_B & I_G^{-1}\,{}_G X_B^{-T} M_{bj}\\
0_{n\times 6} & I_{n\times n}
\end{bmatrix},
$

with the spatial motion transform

$
{}_G X_B =
\begin{bmatrix}
I_{3\times3} & -[x_{b,\mathrm{com}}]_\times\\
0_{3\times3} & I_{3\times3}
\end{bmatrix}.
$

The transformed mass matrix is expected to satisfy

$
M_G = ({}_G T_B)^{-T} M_B ({}_G T_B)^{-1}.
$

##  Imports and robot initialization

In [1]:
import pinocchio as pin
from pinocchio.utils import *
from base_controllers.utils.common_functions import getRobotModel
# console print options to see matrix nicely
np.set_printoptions(precision=3, linewidth=200, suppress=True)
np.set_printoptions(threshold= np.inf)

# Loading a robot model
robot = getRobotModel("solo", generate_urdf = True, floating_base=True)

#start configuration
# set base position
q = pin.neutral(robot.model)
# set joint positions
q[7:] = np.array([
    -0.2,  0.7, -1.4,
    -0.2,  0.7, -1.4,
    -0.2, -0.7,  1.4,
    -0.2, -0.7,  1.4
])


# Generalized velocity: 6 floating-base DoFs + 12 actuated joints
v = np.zeros(robot.model.nv)

# Optional base orientation example from the supplied script
Q = pin.Quaternion(pin.rpy.rpyToMatrix(0.1, 0.1, 0.0))
# q[3:7] = np.array([Q.x, Q.y, Q.z, Q.w])

# Update the joint and frame placements
pin.forwardKinematics(robot.model, robot.data, q, v)
pin.updateFramePlacements(robot.model, robot.data)
robot.computeAllTerms(q, v)

M = pin.crba(robot.model, robot.data, q)
H = pin.nonLinearEffects(robot.model, robot.data, q, v)
G = pin.computeGeneralizedGravity(robot.model, robot.data, q)

print("number of generalized variables (nq) =", robot.model.nq, "number of generalized variables (nv) =", robot.model.nv)
print("Mass matrix shape:", M.shape)

when processing file: /root/ros_ws/install/share/solo_description/robots/solo.urdf.xacro



URDF generated_commons
URDF loaded in Pinocchio
number of generalized variables (nq) = 19 number of generalized variables (nv) = 18
Mass matrix shape: (18, 18)


## Exercise 1 — Center of mass computation (in World frame)

Compute the system CoM as the mass-weighted average of the individual link CoMs,

$
x_{\mathrm{com}} =
\frac{1}{m}\sum_i m_i\,x_{\mathrm{com},i}.
$

Express each link CoM in the world frame before accumulating it. Then compare your result with Pinocchio's `centerOfMass` function.

In [2]:
mass_robot = 0.0
w_com_robot = np.zeros(3)

for idx, joint in enumerate(robot.model.joints):
    if idx == 0:  # skip the first universe link
        continue
    # get mass for link idx
    mass_link = robot.model.inertias[idx].mass
    com_link_local = robot.model.inertias[idx].lever
    # get Homogeneous transform for link idx
    oMi = robot.data.oMi[idx]
    # get com for link idx
    w_com_link = oMi.rotation @ com_link_local + oMi.translation
    # compute total robot mass
    mass_robot += mass_link
    w_com_robot += mass_link * w_com_link
#compute robot com
w_com_robot /= mass_robot
# compute using native pinocchio function
com_pin = pin.centerOfMass(robot.model, robot.data, q, v)

print("Total mass:", mass_robot)
print("CoM from weighted average:", w_com_robot)
print("CoM from Pinocchio:       ", com_pin)
print("Difference:", w_com_robot - com_pin)

assert np.allclose(w_com_robot, com_pin, atol=1e-10)

Total mass: 2.5000027900000004
CoM from weighted average: [ 0.    -0.005 -0.026]
CoM from Pinocchio:        [ 0.    -0.005 -0.026]
Difference: [0. 0. 0.]


## Exercise 2 — Centroidal coordinate transform

Return to the original velocity and compute centroidal quantities with `pin.ccrba`.

Build:

1. the $6\times6$ motion transform ${}_G X_B$;
2. its force transform ${}_G X_B^{-T}$;
3. the joint-to-base coupling block $M_{bj}$;
4. the coupling term
   $
   S_{GB} = I_G^{-1}\,{}_G X_B^{-T} M_{bj};
   $
5. the full generalized velocity transform ${}_G T_B$;
6. verify that the force transform satisfies ${}_G T_B^{-T} = ({}_G T_B^{T})^{-1}$.

In [3]:
# Restore original state
pin.forwardKinematics(robot.model, robot.data, q, v)
pin.updateFramePlacements(robot.model, robot.data)
robot.computeAllTerms(q, v)

M = pin.crba(robot.model, robot.data, q)
g = pin.computeGeneralizedGravity(robot.model, robot.data, q)
com = pin.centerOfMass(robot.model, robot.data, q, v)
# base position in WF
w_base = robot.data.oMi[1].translation.copy()

# Computes centroidal momentum quantities, including robot.data.Ig
pin.ccrba(robot.model, robot.data, q, v)
# number of generalized velocities
nv = robot.model.nv
#number of actuated joints
nj = nv - 6

# compute the motion transform from frame B to framge G (G_X_B)
G_X_B = np.zeros((6, 6))
G_X_B[:3, :3] = np.eye(3)
G_X_B[3:, 3:] = np.eye(3)
G_X_B[:3, 3:] = -pin.skew(com - w_base)

# compute the inverse-transpose of G_X_B (G_X_B^{-T})
G_X_B_invT = np.linalg.inv(G_X_B.T)

# floating base mass submatix
Mbb = M[:6, :6]
#  compute Couplings from joints on the floating base
Mbj = M[:6, 6:]
# Centroidal spatial inertia I_G: locked (6x6) inertia of the whole robot about the CoM
Ig = robot.data.Ig.matrix()
Ig_from_M = G_X_B_invT @ Mbb @ np.linalg.inv(G_X_B)

print("error:",
      np.linalg.norm(Ig - robot.data.Ig.matrix()))

S_G_B = np.linalg.solve(Ig, G_X_B_invT @ Mbj)

# build up the transform from
G_T_B = np.zeros((nv, nv))
G_T_B[:6, :6] = G_X_B
G_T_B[:6, 6:] = S_G_B
G_T_B[6:, 6:] = np.eye(nj)

# double check the transform T has the same propery of the spatial tranforms (G_T_B_invT = inv(G_T_B.T)
G_T_B_invT = np.zeros((nv, nv))
G_T_B_invT[:6, :6] = G_X_B_invT
G_T_B_invT[6:, 6:] = np.eye(nj)
G_T_B_invT[6:, :6] = -S_G_B.T @ G_X_B_invT
G_T_B_invT = np.linalg.inv(G_T_B.T)
print("|| inv(G_T_B^T) - G_T_B_invT || =",
      np.linalg.norm(np.linalg.inv(G_T_B.T) - G_T_B_invT))

assert np.allclose(np.linalg.inv(G_T_B.T), G_T_B_invT, atol=1e-9)

error: 0.0
|| inv(G_T_B^T) - G_T_B_invT || = 0.0


## Exercise 3 — Block-diagonal mass matrix

Transform the inertia matrix:

$
M_G = {}_G T_B^{-T}\, M_B\, ({}_G T_B)^{-1}.
$

Inspect the floating-base/joint coupling blocks. In exact arithmetic they should vanish.

In [4]:

M_G = G_T_B_invT @ M @ np.linalg.inv(G_T_B)
print(M)
coupling_top_right = M_G[:6, 6:]
coupling_bottom_left = M_G[6:, :6]

print("Transformed mass matrix M_G:\n", M_G)
print("\n||top-right coupling||_F =", np.linalg.norm(coupling_top_right))
print("||bottom-left coupling||_F =", np.linalg.norm(coupling_bottom_left))

assert np.allclose(coupling_top_right, 0.0, atol=1e-8)
assert np.allclose(coupling_bottom_left, 0.0, atol=1e-8)

[[ 2.5    0.     0.    -0.    -0.065  0.012  0.    -0.016 -0.003  0.    -0.016 -0.003  0.    -0.016 -0.003  0.    -0.016 -0.003]
 [ 0.     2.5    0.     0.065 -0.     0.     0.018  0.002 -0.     0.018  0.002 -0.     0.015 -0.002  0.     0.015 -0.002  0.   ]
 [ 0.     0.     2.5   -0.012 -0.    -0.     0.005  0.009 -0.002  0.005  0.009 -0.002 -0.011 -0.009  0.002 -0.011 -0.009  0.002]
 [-0.     0.065 -0.012  0.033  0.005 -0.     0.003  0.001 -0.     0.003  0.001 -0.     0.004  0.001 -0.     0.004  0.001 -0.   ]
 [-0.065 -0.    -0.     0.005  0.069 -0.001 -0.001  0.001  0.001  0.001  0.005  0.     0.003  0.005  0.    -0.002  0.001  0.001]
 [ 0.012  0.    -0.    -0.    -0.001  0.083  0.003  0.002  0.    -0.004  0.001  0.     0.004 -0.003 -0.    -0.002 -0.002 -0.001]
 [ 0.     0.018  0.005  0.003 -0.001  0.003  0.003  0.    -0.     0.     0.     0.     0.     0.     0.     0.     0.     0.   ]
 [-0.016  0.002  0.009  0.001  0.001  0.002  0.     0.003  0.001  0.     0.     0.     0.     0. 

## Exercise 4 — Gravity vector in CoM coordinates

The lab asks you to check the centroidal gravity structure


$
\begin{equation*}
 g_G={}_G T_B^{-T}g_B=
\begin{bmatrix}
0\\0\\mg\\0_{3\times1}\\0_{n\times1}
\end{bmatrix}.
\end{equation*}
$


In [5]:
g_world = np.array([0.0, 0.0, robot.robotMass*9.81])
expected_g = np.hstack((g_world, np.zeros(nv - 3)))

# comopute centoidal gravity vector
g_G = G_T_B_invT.dot(g)

expected = np.zeros(nv)
expected[2] = mass_robot * 9.81
print( "\n The gravity force vector at the com should be  [0 0 mg   03x1    0nx1 ]: \n", g_G)
print("\nDifference:\n", g_G - expected_g)


 The gravity force vector at the com should be  [0 0 mg   03x1    0nx1 ]: 
 [ 0.     0.    24.525  0.     0.    -0.     0.     0.     0.     0.    -0.     0.    -0.    -0.    -0.    -0.    -0.    -0.   ]

Difference:
 [ 0.  0. -0.  0.  0. -0.  0.  0.  0.  0. -0.  0. -0. -0. -0. -0. -0. -0.]
